In [1]:
import random
import sys

import numpy as np

sys.path.insert(0, "../../lib") 
from neurality import Neurality as N # type: ignore

# Seed agar hasil training dan dropout reproducible.
random.seed(0)
np.random.seed(0)

text = open("cerita.txt", "r").read().replace("\n", " ").strip()[0:1000]


# Byte-level BPE: mulai dari 256 byte, lalu gabungkan pasangan terpopuler.
# Vocabulary kecil karena corpus contoh ini hanya satu paragraf.
class BytePairTokenizer:
    def __init__(self):
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = []

    @staticmethod
    def merge(tokens, pair, new_id):
        result = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair:
                result.append(new_id)
                i += 2
            else:
                result.append(tokens[i])
                i += 1
        return result

    def train(self, text, vocab_size=288):
        self.__init__()
        tokens = list(text.encode("utf-8"))
        while len(self.vocab) < vocab_size:
            counts = {}
            for pair in zip(tokens, tokens[1:]):
                counts[pair] = counts.get(pair, 0) + 1
            if not counts:
                break
            pair = max(counts, key=counts.get)
            if counts[pair] < 2:
                break
            new_id = len(self.vocab)
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            self.merges.append((pair, new_id))
            tokens = self.merge(tokens, pair, new_id)
        return self

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for pair, new_id in self.merges:
            tokens = self.merge(tokens, pair, new_id)
        return tokens

    def decode(self, tokens):
        return b"".join(self.vocab[int(token)] for token in tokens).decode("utf-8", errors="replace")


tokenizer = BytePairTokenizer().train(text)
tokens = tokenizer.encode(text)
vocab_size = len(tokenizer.vocab)
context_length = len(tokens) - 1
assert tokenizer.decode(tokens) == text
print(f"Byte: {len(text.encode('utf-8'))}, token BPE: {len(tokens)}, vocabulary: {vocab_size}")

Byte: 1000, token BPE: 647, vocabulary: 288


In [2]:
text

'Hujan baru saja reda. Lina datang ke sekolah lama dengan membawa surat yang belum terkirim. Ia sudah memikirkan rencana itu sejak beberapa hari lalu, tetapi belum sempat membicarakannya dengan siapa pun.  Di sana, Lina bertemu Fajar. Mereka duduk berdekatan dan mulai mencatat surat yang belum terkirim dengan teliti. Sesekali, Fajar mengajukan pertanyaan yang membuat Lina berhenti sejenak untuk berpikir.  "Menurutmu, dari mana kita harus mulai?" tanya Lina.  "Kita perlu memahami persoalannya lebih dulu," jawab Fajar. "Kalau langsung mengambil keputusan, mungkin ada hal penting yang terlewat."  Mereka kemudian mengamati beberapa kemungkinan. Lina menuliskan gagasan utama pada selembar kertas, sementara Fajar mencoba mencari contoh yang mudah dipahami. Percakapan mereka berlangsung tenang, meskipun pendapat keduanya tidak selalu sama.  Ketika hari mulai berganti, mereka menyepakati langkah kecil yang dapat dilakukan esok. Lina pulang dengan perasaan lega. Masalah itu memang belum selesai

In [3]:
# Input dan target digeser satu token BPE.
inputs = [tokens[:-1]]
next_tokens = tokens[1:]

# CCE membutuhkan target one-hot
targets = [[
    [float(i == token) for i in range(vocab_size)]
    for token in next_tokens
]]

dropout_rate = 0.1

model = N(backend=N.Backends.NUMPY)

model.embedding(vocab_size=vocab_size, embedding_size=16)
model.positional_embedding(max_length=context_length)
model.dropout(dropout_rate)

model.residual_start()
model.norm()
model.multihead_attention(num_heads=4, causal=True)
model.dropout(dropout_rate)
model.residual_end()

model.residual_start()
model.norm()
model.dense(64, N.Activations.gelu)
model.dense(16, N.Activations.linear)
model.dropout(dropout_rate)
model.residual_end()

model.norm()
model.dense(vocab_size, N.Activations.softmax)
model.optimizer(N.Optimizers.Adam(learning_rate=0.01))

# Dropout aktif hanya saat training; eval() mematikannya.
model.train()

# Latih untuk memprediksi token BPE berikutnya
for epoch in range(350):
    prediction = model.forward(inputs)
    loss = N.Loss.loss_cce(prediction, targets)
    model.backward(targets, N.Gradients.gradient_cce)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}: loss = {loss:.4f}")

Epoch 50: loss = 0.3630
Epoch 100: loss = 0.0454
Epoch 150: loss = 0.0200
Epoch 200: loss = 0.0081
Epoch 250: loss = 0.0070
Epoch 300: loss = 0.0068
Epoch 350: loss = 0.0051


In [ ]:
def sample_token(probabilities, temperature=1.0):
    """Sampling dari output softmax; temperature 0 memakai argmax."""
    if not np.isfinite(temperature) or temperature < 0:
        raise ValueError("Temperature harus finite dan >= 0")

    probabilities = np.asarray(probabilities, dtype=np.float64)
    if temperature == 0:
        return int(np.argmax(probabilities))

    # Output model sudah softmax: ubah ke log-probabilitas sebelum scaling.
    logits = np.log(np.clip(probabilities, 1e-300, 1.0))
    logits = (logits - np.max(logits)) / temperature
    adjusted = np.exp(logits)
    adjusted /= adjusted.sum()

    return int(np.random.choice(len(adjusted), p=adjusted))


# Generasi autoregresif: prediksi satu token BPE pada setiap langkah.
prompt = "Kita perlu memahami persoalannya lebih dulu"
generated = tokenizer.encode(prompt)
temperature = 0.8  # 0: argmax, <1: lebih terarah, 1: distribusi asli, >1: lebih acak

model.eval()  # matikan dropout saat inference

for _ in range(1000):
    # Batasi konteks agar tidak melampaui kapasitas positional embedding.
    context = generated[-context_length:]
    probabilities = model.forward([context])[0][-1]
    next_id = sample_token(probabilities, temperature=temperature)
    generated.append(next_id)

print("Hasil:", tokenizer.decode(generated))

In [ ]:
print("Token prompt:", tokenizer.encode(prompt))
print("Potongan token:", [tokenizer.vocab[i] for i in tokenizer.encode(prompt)])
print("Hasil:", repr(tokenizer.decode(generated)))
